# T10 — DeBERTa-v3-base on Colab

`deberta-v3-base` cannot train on the local RTX 3050: fp32 needs 2.94 GB for weights,
grads and two AdamW moments alone, on 3.65 GB usable, and every batch size OOMs even
with gradient checkpointing. A T4 (15 GB) has room.

**Push the local repo before running this.** Two commits are required:

- `179a678` — `--model` override, or the encoder cannot be selected
- `57f1b78` — forces fp32. **Without it DeBERTa NaNs on the first optimizer step**,
  because its checkpoint ships fp16 and AdamW's `denom` underflows to zero.

Cell 2 asserts both are present rather than letting you find out at epoch 1.

ACTER is cloned from its public upstream at `f05b09e` (v1.5), the same commit as local.
Nothing is uploaded and nothing non-commercial is redistributed.

**Runtime → Change runtime type → T4 GPU** first.


In [ ]:
# 1. Colab-only guard, GPU check, and the command helper used below.
try:
    import google.colab  # noqa: F401
except ImportError:
    raise SystemExit(
        'This notebook runs on Google Colab only.\n'
        'Open it at colab.research.google.com -> File -> Upload notebook,\n'
        'or File -> Open notebook -> GitHub -> ahmedwaleedaref/ATE-ACTER.')

import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'
print('torch', torch.__version__, '| cuda', torch.version.cuda)

def run(*args):
    """Stream a command into the cell, raise on failure. Plain subprocess:
    a multi-line ! with continuations inside a loop is invalid after IPython's
    transform."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(args)}')


In [ ]:
# 2. Clone the project and the corpus
%cd /content
!rm -rf ate-acter
!git clone -q https://github.com/ahmedwaleedaref/ATE-ACTER.git ate-acter
%cd /content/ate-acter
!git log --oneline -1
!git clone -q https://github.com/AylaRT/ACTER.git data/raw/ACTER
!cd data/raw/ACTER && git checkout -q f05b09e985cad37eeaa8daa8b3f383197aa5324e

import subprocess
log = subprocess.run(['git','log','--oneline','-60'], capture_output=True, text=True).stdout
assert '--model override' in log, 'repo predates --model: push 179a678 first'
assert 'forces fp32'   in log, 'repo predates the fp32 fix: push 57f1b78, or DeBERTa will NaN'
assert __import__('pathlib').Path('data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print('\nrepo + corpus OK')


In [ ]:
# 3. Pinned installs.
#
# torch/numpy/scikit-learn are deliberately NOT pinned: Colab ships a torch built
# for its own driver, and forcing the local 2.14.0 pulls a mismatched CUDA build.
# The run JSON records the torch version, so the deviation stays in the record.
!pip install transformers==5.16.1 tokenizers==0.23.1 safetensors==0.8.0 \
             huggingface_hub==1.29.0 sentencepiece==0.2.2 \
             protobuf==7.36.0 PyYAML==6.0.3 pytest==8.3.2

# seqeval ships an sdist only, and its legacy setup.py fails `egg_info` under
# recent setuptools. It is NOT on the training path: the sole importer is
# tests/test_seqeval_agreement.py, which cross-checks score_exact_spans against
# seqeval. Best-effort, and the run is unaffected if it will not build.
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0

HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('\nseqeval:', 'installed (47 tests)' if HAVE_SEQEVAL else
      'UNAVAILABLE -- 46 of 47 tests will run; training is unaffected')


In [ ]:
# 4. Environment check. A version gap is a T10 confound -- record it in E04.
import importlib
for mod in ('torch', 'transformers', 'tokenizers', 'numpy'):
    print(f'{mod:14} {importlib.import_module(mod).__version__}')
print('python', sys.version.split()[0], ' (local: 3.14.4, torch 2.14.0, transformers 5.16.1)')
print()
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
if skip:
    print('NOTE: test_seqeval_agreement.py deselected -- seqeval did not install.')
    print('      score_exact_spans is then unverified against seqeval on this machine.')
run(sys.executable, '-m', 'pytest', 'tests/', '-q', *skip)


In [ ]:
# 5. Single-seed probe. A finite train_loss at epoch 1 means the fp32 fix is live.
import json, pathlib
TRAIN = [sys.executable, '-m', 'src.models.run_train',
         '--model', 'microsoft/deberta-v3-base',
         '--group', 't10/deberta-v3-base',
         '--reason', 'T10 deberta-v3-base, T9 config, Colab T4']

def summarise(seed):
    """Echo the numbers into the notebook output, which is saved with the
    notebook. If the Colab session dies, /content goes with it -- this is the
    copy that survives."""
    r = json.loads(pathlib.Path(f'results/runs/t10/deberta-v3-base/seed_{seed}.json').read_text())
    print(f"  SEED {seed}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
          f"htfl={r['htfl_f1']:.4f} collapsed={r['collapsed']} "
          f"torch={r['versions']['torch']} {r['wall_time_sec']}s")

run(*TRAIN, '--seed', '42'); summarise(42)


In [ ]:
# 6. Remaining four seeds. Run only after the probe above looks sane.
#    Expect ~10-15 min each on a T4; a free session may not survive all four.
for s in (43, 44, 45, 46):
    print(f'\n===== seed {s} =====')
    run(*TRAIN, '--seed', str(s))
    summarise(s)


In [ ]:
# 7. Cell stats, then download. --cell needs all five seeds present.
run(sys.executable, '-m', 'src.aggregate', '--cell', 'results/runs/t10/deberta-v3-base')

!zip -qr /content/t10_deberta.zip results/runs/t10 results/test_evaluations.log
from google.colab import files
files.download('/content/t10_deberta.zip')
